# Stage 2 — bootstrapping a ESTR OIS discount curve

Stage 1 took observed yields and fitted a smooth shape through them. Stage 2 does the opposite: it takes tradeable swap quotes and solves out discount factors **exactly**, one at a time, with no model at all.

This is the curve a rates desk actually discounts with. Since the 2008–2010 shift away from LIBOR discounting, collateralised euro derivatives are discounted at the overnight rate — ESTR — so the ESTR OIS curve is *the* euro discount curve.

By the end you should be able to:

1. derive why an OIS floating leg is worth exactly `1 - DF(T)`, and see why that makes the bootstrap trivial,
2. bootstrap a curve with pencil and paper for the first two pillars, then let code do the rest,
3. explain what your interpolation choice does to forward rates, and why that is the only place to look,
4. compute a bucketed delta ladder — the risk report the whole trading floor reads.

In [1]:
import sys, pathlib, datetime as dt
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eurocurve import ecb
from eurocurve.curve import DiscountCurve
from eurocurve.bootstrap import OISSwap, bootstrap_ois, load_quotes, par_rate, reprice_check, spot_date
from eurocurve.daycount import year_fraction, annual_schedule

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA = pathlib.Path.cwd().parent / 'data' / 'eur_ois_sample.csv'

## 1. The instrument

A EUR OIS: you pay a fixed rate `S`, you receive ESTR compounded daily, over the same period. Conventions:

| | |
|---|---|
| index | ESTR (euro short-term rate, published by the ECB each morning for the previous day) |
| fixed frequency | annual beyond 1Y; single payment at maturity for 1Y and shorter |
| day count | ACT/360, both legs |
| start | spot, T+2 business days |
| roll | modified following, TARGET2 calendar |

Live OIS quotes are available on Bloomberg / Refinitiv / ICAP. Here, we have a artificaial EUR curve `data/eur_ois_sample.csv`  which makes this notebook reproducible offline. The ESTR fixing itself *is* free, so it is pulled live and use it to anchor the overnight end. The bootstrap arithmetic is identical with real or artificial quotes as long as the quote file format is maintained.

In [2]:
valuation = dt.date.today()
spot = spot_date(valuation)
print('valuation', valuation, '| spot (T+2)', spot)

estr_date, estr = ecb.fetch_estr()
print(f'ESTR {estr:.4%} as of {estr_date}  (live from the ECB)')

swaps = load_quotes(DATA, spot=spot)
pd.DataFrame({'tenor': [s.tenor for s in swaps],
              'rate_%': [s.rate*100 for s in swaps],
              'maturity_y': [round(s.maturity, 3) for s in swaps],
              'n_payments': [len(s.pay_dates) for s in swaps]})

valuation 2026-08-19 | spot (T+2) 2026-08-21
fetching URL: https://data-api.ecb.europa.eu/service/data/EST/B.EU000A2X2A25.WT
ESTR 2.1880% as of 2026-08-18  (live from the ECB)


,tenor,rate_%,maturity_y,n_payments
0,1W,1.930,0.019,1
1,1M,1.928,0.085,1
2,3M,1.925,0.258,1
3,6M,1.940,0.507,1
4,9M,1.970,0.748,1
5,1Y,2.005,1.005,1
6,18M,2.090,1.504,2
7,2Y,2.170,2.003,2
8,3Y,2.290,3.003,3
9,4Y,2.375,4.003,4


#### Note on schedule generation

Schedules are generated **backwards from maturity**, because maturity is the date both counterparties agreed on and the stub goes at the front. If forwards are generated from the start date instead, we get a  plausible-looking curve that is quietly wrong. The pytest `tests/test_daycount.py::test_broken_tenor_generates_backwards` catches this. In particular for the 18m swap above: the schedule is at 6M and 18M — not 12M and 18M. 


In [3]:
s18 = next(s for s in swaps if s.tenor == '18M')
for d, a in zip(s18.pay_dates, s18.alphas):
    print(f'{d}   ~{(d-spot).days/365:.2f}y after spot   accrual {a:.6f}')

2027-02-22   ~0.51y after spot   accrual 0.513889
2028-02-21   ~1.50y after spot   accrual 1.011111


## 2. Swap legs' PV calculation

The swap is worth zero at inception, so `PV(fixed) = PV(float)`:

```
S * sum_i alpha_i * DF(t_i)  =  PV(float)
```

The left side is the fixed rate times the **annuity**. Now the right side. Take one floating period. In a single-curve world — where the rate you project is the same rate you discount with, which for ESTR OIS is exactly true — the forward compounded rate over `[t_{i-1}, t_i]` satisfies

```
1 + alpha_i * F_i = DF(t_{i-1}) / DF(t_i)
```

so the PV of that one payment is

```
alpha_i * F_i * DF(t_i) = (DF(t_{i-1})/DF(t_i) - 1) * DF(t_i) = DF(t_{i-1}) - DF(t_i)
```

Sum over all periods. Every interior term appears once positive and once negative. It results in the classic formula for swaps pricing:

```
PV(float) = DF(t_0) - DF(t_n) = 1 - DF(T)
```

The floating leg is worth "a euro now minus a euro at maturity", whatever the forwards do. Which hands us the par rate:

```
S(T) = (1 - DF(T)) / sum_i alpha_i * DF(t_i)
```

Numerically, on an arbitrary made-up curve, here is the trivial comparison.

In [4]:
rng = np.random.default_rng(3)
tt = np.arange(1, 11.0)
junk = DiscountCurve.from_zero_rates(tt, 0.02 + rng.normal(0, 0.004, tt.size))

dfs = np.concatenate([[1.0], np.asarray(junk.df(tt))])
alphas = np.full(10, 1.0)
fwds = (dfs[:-1]/dfs[1:] - 1)/alphas          # forward for each period
pv_the_hard_way = float(np.sum(alphas*fwds*dfs[1:]))
pv_the_easy_way = 1.0 - float(junk.df(10.0))

print(f'sum of discounted forward payments : {pv_the_hard_way:.15f}')
print(f'1 - DF(10)                         : {pv_the_easy_way:.15f}')
print(f'difference                         : {pv_the_hard_way - pv_the_easy_way:.2e}')

sum of discounted forward payments : 0.283172448075753
1 - DF(10)                         : 0.283172448075752
difference                         : 3.33e-16


Identical to machine precision, on a test curve. The result is structural, not numerical.

## 3. Bootstrapping by hand

Now walk up the maturities. The 1Y swap has a **single** payment, so its equation has one unknown:

```
S * alpha_1 * DF(T1) = 1 - DF(T1)
      =>  DF(T1) = 1 / (1 + S * alpha_1)
```

The 2Y swap pays at 1Y and 2Y. We know `DF(T1)`, so again one equation, one unknown:

```
DF(T2) = (1 - S2*alpha_1*DF(T1)) / (1 + S2*alpha_2)
```

This is the bootstrapping process — each instrument contributes exactly one new pillar. First two are done with plain arithmetic and checked against the bootstrapping code.

In [5]:
s1 = next(s for s in swaps if s.tenor == '1Y')
s2 = next(s for s in swaps if s.tenor == '2Y')

a1 = s1.alphas[0]
df1 = 1.0 / (1.0 + s1.rate*a1)

b1, b2 = s2.alphas
df2 = (1.0 - s2.rate*b1*df1) / (1.0 + s2.rate*b2)

print(f'by hand : DF(1Y) = {df1:.12f}   DF(2Y) = {df2:.12f}')

lib_curve = bootstrap_ois([s1, s2])
print(f'library : DF(1Y) = {lib_curve.df(s1.maturity):.12f}   DF(2Y) = {lib_curve.df(s2.maturity):.12f}')
print(f'\nimplied 1y zero (annual comp): {lib_curve.zero(s1.maturity, "annual"):.4%}')
print(f'quoted 1y OIS par rate       : {s1.rate:.4%}')
print('the gap is the ACT/360 vs ACT/365 basis, ~1.4% of the rate — not an error')

by hand : DF(1Y) = 0.979969558335   DF(2Y) = 0.957316578457
library : DF(1Y) = 0.979969558335   DF(2Y) = 0.957316578457

implied 1y zero (annual comp): 2.0327%
quoted 1y OIS par rate       : 2.0050%
the gap is the ACT/360 vs ACT/365 basis, ~1.4% of the rate — not an error


## 4. Bootstrap curve from swap quotes

The quote set jumps `10Y → 12Y → 15Y → 20Y → 25Y → 30Y`, so the 15Y swap pays at 11y, 13y and 14y where we have nothing.

For each swap, find the `DF(T)` such that the curve reprices the swap at its quote, letting the intermediate discount factors come from interpolating between the last known pillar and the candidate. `scipy.optimize.brentq` is used for root-solving. The interpolation scheme is part of the curve's definition and determines the value of the bootstrapped pillars themselves.

In [8]:
print(len(swaps))

curve = bootstrap_ois(swaps, overnight_rate=estr, as_of=valuation)
print(curve)

21
DiscountCurve(23 pillars, 0.003y-30.02y, as_of=2026-08-19)


In [10]:
chk = reprice_check(swaps, curve)
print(f"\nworst repricing error: {chk['error_bp'].abs().max():.3e} bp")
chk.round(8)


worst repricing error: 3.237e-11 bp


,tenor,quoted_pct,reprice_pct,error_bp,pv_per_100m
0,1W,1.930,1.930,-0.0,1.000000e-08
1,1M,1.928,1.928,0.0,-1.000000e-08
2,3M,1.925,1.925,-0.0,3.000000e-08
3,6M,1.940,1.940,0.0,-1.000000e-08
4,9M,1.970,1.970,-0.0,1.000000e-08
5,1Y,2.005,2.005,-0.0,1.000000e-08
6,18M,2.090,2.090,0.0,-0.000000e+00
7,2Y,2.170,2.170,0.0,-0.000000e+00
8,3Y,2.290,2.290,0.0,-1.000000e-08
9,4Y,2.375,2.375,-0.0,0.000000e+00


Comparing use of swap quotes to build a bootstrapped curve vs. parametric fit for the curve:

| | bootstrap | parametric fit |
|---|---|---|
| reprices inputs | exactly | approximately |
| degrees of freedom | one per instrument | six, total |
| absorbs quote noise | no — it enshrines it | yes |
| parameters mean anything | no | yes, loosely |
| use it for | marking and hedging a book | comparing curves across time or country |
